# CBIS-DDSM — 3-input training pipeline (full image + cropped ROI + ROI mask)

This mirrors `model_training_pipeline_full_photo.ipynb`, but each sample now feeds
**three** images into the model instead of one:

1. the full mammogram (global context)
2. the cropped ROI patch (zoomed-in lesion detail)
3. the ROI mask (tells the model exactly where the lesion is)

**Expected folder layout** (this notebook lives at `experiment/notebooks/`):
```
experiment/
├── notebooks/
│   └── model_training_pipeline_three_input.ipynb   <- this notebook
└── dataframes/
    ├── three_input_train_df.csv
    ├── three_input_val_df.csv
    └── three_input_test_df.csv
```

The dataframes contain **three directory-path columns** (full/crop/mask), where each
path points to a folder containing PNG files — the actual PNG filenames are not
stored in the dataframe, so this notebook discovers them on disk at load time.

For CBIS-DDSM, the cropped-image and ROI-mask columns commonly point to the **same**
folder, so this notebook resolves crop-vs-mask by looking inside that shared folder
and disambiguating by image area: smallest image is treated as crop, largest image
as mask.

Run this notebook cell-by-cell the first time and verify the visual sanity-check outputs
before launching full multi-model training.

In [16]:
import os
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset


def win_long_path(p: Path) -> str:
    """Return a path string safe for Windows' 260-char MAX_PATH limit."""
    p = p.resolve()
    s = str(p)
    if os.name == "nt" and not s.startswith("\\\\?\\"):
        s = "\\\\?\\" + s
    return s


# This notebook lives at experiment/notebooks/<this notebook>.ipynb, so PROJECT_ROOT
# (one level up) resolves to experiment/. That's also where dataframes/ lives:
#   experiment/dataframes/three_input_train_df.csv
#   experiment/dataframes/three_input_val_df.csv
#   experiment/dataframes/three_input_test_df.csv
PROJECT_ROOT = Path("..").resolve()
DATAFRAMES_ROOT = PROJECT_ROOT / "dataframes"
PNG_ROOT = PROJECT_ROOT / "cbis_ddsm_png"
CACHE_ROOT = PROJECT_ROOT / "cbis_ddsm_cache"
# Slightly larger than the 224 the model trains on, so RandomRotation-style
# augmentation still has a little margin to work with (matches the full-image cache).
CACHE_SIZE = 256

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATAFRAMES_ROOT:", DATAFRAMES_ROOT)
print("PNG_ROOT:", PNG_ROOT)

## Resolving the crop + mask files

CBIS-DDSM's `cropped image file path` and `ROI mask file path` columns frequently
point to the **same** folder, which then contains 2 PNGs with no reliable way to
tell them apart from the filename alone. We disambiguate by size instead: the ROI
mask is (approximately) full-mammogram-sized, while the crop is a small, tight patch
around the lesion — so whichever file has the larger area is the mask, and whichever
is smaller is the crop.

If the two path columns resolve to different folders, we just trust that separation
directly (falling back to the size heuristic only if a folder unexpectedly contains
more than one PNG).

In [ ]:
def _to_abs_png_dir(path_value):
    """Normalize one dataframe path value into an absolute PNG directory path."""
    if pd.isna(path_value):
        raise ValueError("Encountered NaN in a path column.")

    raw = str(path_value).strip().replace("\\", "/")
    if raw == "":
        raise ValueError("Encountered empty string in a path column.")

    p = Path(raw)
    if p.is_absolute():
        return p

    # If dataframe already includes cbis_ddsm_png/... anchor it to project root.
    if raw.startswith("cbis_ddsm_png/") or raw.startswith("cbis_ddsm_png\\") or raw.startswith("cbis_ddsm_png"):
        return PROJECT_ROOT / p

    # Otherwise treat it as dataset-relative and anchor under PNG_ROOT.
    return PNG_ROOT / p


def _pick_single_png(png_dir):
    candidates = sorted(png_dir.rglob("*.png"))
    if len(candidates) == 0:
        raise FileNotFoundError(f"No PNG files found under:\n{png_dir}")
    return candidates[0]


def _resolve_crop_and_mask(crop_dir_value, mask_dir_value):
    """Return (crop_path, mask_path) from row directory values.

    CBIS-DDSM's `cropped image file path` and `ROI mask file path` columns
    normally point to the *same* folder, which then holds two PNGs with no
    reliable way to tell them apart from the filename alone. We treat that as
    the default case: resolve both column values to real directories first
    (`.resolve()`, so trailing slashes / `..` segments / relative-vs-absolute
    formatting differences introduced by the new dataframe don't break the
    comparison), and whenever they land on the same folder, gather every PNG
    inside it and disambiguate by image area: the ROI mask is (approximately)
    full-mammogram-sized, while the crop is a small, tight patch around the
    lesion, so the smallest-area file is the crop and the largest-area file
    is the mask.

    If the two columns genuinely resolve to different folders, we trust that
    separation directly, falling back to the size heuristic only if one of
    those folders unexpectedly contains more than one PNG."""
    crop_dir = _to_abs_png_dir(crop_dir_value).resolve()
    mask_dir = _to_abs_png_dir(mask_dir_value).resolve()

    if crop_dir == mask_dir:
        candidates = sorted(crop_dir.rglob("*.png"))
        if len(candidates) == 0:
            raise FileNotFoundError(f"No PNG files found under:\n{crop_dir}")
        if len(candidates) == 1:
            # Best-effort fallback when only one file exists.
            return candidates[0], candidates[0]

        sized = []
        for p in candidates:
            with Image.open(win_long_path(p)) as im:
                sized.append((p, im.size[0] * im.size[1]))
        sized.sort(key=lambda t: t[1])
        crop_path = sized[0][0]   # smallest area -> crop
        mask_path = sized[-1][0]  # largest area  -> ROI mask
        return crop_path, mask_path

    crop_path = _pick_single_png(crop_dir)
    mask_path = _pick_single_png(mask_dir)
    return crop_path, mask_path


PATH_COLUMN_ALIASES = {
    "full": [
        "image file path",
        "full image file path",
        "full_image_file_path",
        "full_path",
    ],
    "crop": [
        "cropped image file path",
        "crop image file path",
        "cropped_image_file_path",
        "crop_path",
    ],
    "mask": [
        "ROI mask file path",
        "roi mask file path",
        "roi_mask_file_path",
        "mask_path",
    ],
}


def infer_path_columns(df):
    resolved = {}
    for key, candidates in PATH_COLUMN_ALIASES.items():
        found = next((c for c in candidates if c in df.columns), None)
        if found is None:
            raise KeyError(
                f"Could not find a '{key}' path column. Tried: {candidates}. "
                f"Available columns: {df.columns.tolist()}"
            )
        resolved[key] = found
    return resolved


class CBISThreeInputRawDataset(Dataset):
    """Resolves valid (full, crop, mask) paths per row and skips unresolved rows.
    This keeps triplets aligned while making the pipeline robust to missing PNGs."""

    def __init__(self, dataframe, path_cols):
        df = dataframe.reset_index(drop=True)
        self.path_cols = path_cols
        self.full_paths, self.crop_paths, self.mask_paths = [], [], []
        self._kept_row_idx = []
        self._skipped = []

        n = len(df)
        for idx, row in df.iterrows():
            try:
                full_dir = _to_abs_png_dir(row[self.path_cols["full"]])
                full_path = _pick_single_png(full_dir)

                crop_path, mask_path = _resolve_crop_and_mask(
                    row[self.path_cols["crop"]], row[self.path_cols["mask"]]
                )

                self.full_paths.append(full_path)
                self.crop_paths.append(crop_path)
                self.mask_paths.append(mask_path)
                self._kept_row_idx.append(idx)
            except (FileNotFoundError, OSError, ValueError) as exc:
                self._skipped.append((idx, str(exc)))

            if (idx + 1) % 200 == 0 or (idx + 1) == n:
                print(f"resolved {idx + 1}/{n} rows")

        self.df = df.iloc[self._kept_row_idx].reset_index(drop=True)

        if self._skipped:
            print(f"Skipped {len(self._skipped)} rows with unresolved image paths.")
            print("First skipped examples:")
            for bad_idx, msg in self._skipped[:5]:
                print(f"  - row {bad_idx}: {msg.splitlines()[0]}")

    def __len__(self):
        return len(self.df)

In [ ]:
import pandas as pd

train_df = pd.read_csv(DATAFRAMES_ROOT / "three_input_train_df.csv")
val_df = pd.read_csv(DATAFRAMES_ROOT / "three_input_val_df.csv")
test_df = pd.read_csv(DATAFRAMES_ROOT / "three_input_test_df.csv")

print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)
print("columns:", train_df.columns.tolist())

path_cols = infer_path_columns(train_df)
print("resolved path columns:", path_cols)

# --- Quick diagnostic on a single row before resolving the whole dataframe ---
# Confirms the new folder structure resolves correctly (and shows exactly which
# PNGs get picked up as full/crop/mask) before we run this over every row.
_probe_row = train_df.iloc[0]
_full_dir = _to_abs_png_dir(_probe_row[path_cols["full"]])
_crop_dir = _to_abs_png_dir(_probe_row[path_cols["crop"]]).resolve()
_mask_dir = _to_abs_png_dir(_probe_row[path_cols["mask"]]).resolve()

print("\nRow 0 diagnostic:")
print("  full dir:", _full_dir, "-> pngs:", [p.name for p in sorted(_full_dir.rglob('*.png'))])
print("  crop dir:", _crop_dir)
print("  mask dir:", _mask_dir)
print("  crop/mask share a folder:", _crop_dir == _mask_dir)
_shared_pngs = sorted(_crop_dir.rglob('*.png')) if _crop_dir == _mask_dir else (
    sorted(_crop_dir.rglob('*.png')) + sorted(_mask_dir.rglob('*.png'))
)
print("  pngs found:", [p.name for p in _shared_pngs])

_probe_full, _probe_crop, _probe_mask = _pick_single_png(_full_dir), *_resolve_crop_and_mask(
    _probe_row[path_cols["crop"]], _probe_row[path_cols["mask"]]
)
print("  -> full:", _probe_full.name, "| crop:", _probe_crop.name, "| mask:", _probe_mask.name)

# --- Now resolve every row for all three splits ---
raw_train_source = CBISThreeInputRawDataset(train_df, path_cols)
raw_val_source = CBISThreeInputRawDataset(val_df, path_cols)
raw_test_source = CBISThreeInputRawDataset(test_df, path_cols)

### Visual sanity check (do this before caching everything)

Open one resolved triplet and actually look at it — confirm the "crop" is a small
zoomed patch and the "mask" looks like a binary blob roughly the shape/size of the
full mammogram. If they're swapped, the size heuristic above picked the wrong file
for at least this dataset copy and needs adjusting (e.g. some sources store masks
with a different aspect ratio than the full image, which breaks the "largest area"
assumption).

In [ ]:
sample_idx = 0

full_img = Image.open(win_long_path(raw_train_source.full_paths[sample_idx]))
crop_img = Image.open(win_long_path(raw_train_source.crop_paths[sample_idx]))
mask_img = Image.open(win_long_path(raw_train_source.mask_paths[sample_idx]))

print("full :", full_img.size)
print("crop :", crop_img.size)
print("mask :", mask_img.size)

full_img

In [ ]:
crop_img

In [ ]:
mask_img

## Build the cache for all splits (full + crop + mask)

Each split (`train`, `val`, `test`) is cached into flat `{idx}.png` files for fast I/O.
This cell builds all three modalities per split:

- full image cache: `cbis_ddsm_cache/{split}/`
- crop cache: `cbis_ddsm_cache/{split}_crop/`
- ROI mask cache: `cbis_ddsm_cache/{split}_mask/`

Masks are resized with **nearest-neighbor** interpolation to preserve binary edges.

In [ ]:
def build_three_input_cache(raw_source, split_name):
    """Resize and cache full/crop/mask images for one split.
    Safe to re-run because existing cache files are skipped."""
    full_out_dir = CACHE_ROOT / split_name
    crop_out_dir = CACHE_ROOT / f"{split_name}_crop"
    mask_out_dir = CACHE_ROOT / f"{split_name}_mask"

    full_out_dir.mkdir(parents=True, exist_ok=True)
    crop_out_dir.mkdir(parents=True, exist_ok=True)
    mask_out_dir.mkdir(parents=True, exist_ok=True)

    n = len(raw_source)
    for idx in range(n):
        full_out_path = full_out_dir / f"{idx}.png"
        crop_out_path = crop_out_dir / f"{idx}.png"
        mask_out_path = mask_out_dir / f"{idx}.png"

        if not full_out_path.exists():
            full_image = Image.open(win_long_path(raw_source.full_paths[idx])).convert("RGB")
            full_image = full_image.resize((CACHE_SIZE, CACHE_SIZE), Image.BILINEAR)
            full_image.save(full_out_path)

        if not crop_out_path.exists():
            crop_image = Image.open(win_long_path(raw_source.crop_paths[idx])).convert("RGB")
            crop_image = crop_image.resize((CACHE_SIZE, CACHE_SIZE), Image.BILINEAR)
            crop_image.save(crop_out_path)

        if not mask_out_path.exists():
            mask_image = Image.open(win_long_path(raw_source.mask_paths[idx])).convert("RGB")
            mask_image = mask_image.resize((CACHE_SIZE, CACHE_SIZE), Image.NEAREST)
            mask_image.save(mask_out_path)

        if (idx + 1) % 200 == 0 or (idx + 1) == n:
            print(f"[{split_name}] cached {idx + 1}/{n}")


build_three_input_cache(raw_train_source, "train")
build_three_input_cache(raw_val_source, "val")
build_three_input_cache(raw_test_source, "test")

print("\nThree-input caching done for train/val/test.")

## Cached dataset with synchronized augmentation and channel stacking

The full image, crop, and mask are spatially related for each row, so augmentation
(flip/rotation) is sampled once and applied identically to all three modalities.

After augmentation, each modality is converted to **single-channel grayscale** and
stacked in a fixed channel order into one 3-channel tensor:

- Channel 0: full mammogram
- Channel 1: cropped ROI
- Channel 2: ROI mask

This gives one ordinary 3-channel input per sample while preserving the relationship
between all three image types.

In [ ]:
import random

from torchvision.transforms import functional as TF

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

RESIZE_SIZE = 224
ROTATION_DEGREES = 10


class CBISThreeInputCachedDataset(Dataset):
    """Reads cached full/crop/mask images and returns one stacked 3-channel tensor.
    Channel order is fixed: [full, crop, mask]."""

    def __init__(self, dataframe, split_name, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.full_dir = CACHE_ROOT / split_name
        self.crop_dir = CACHE_ROOT / f"{split_name}_crop"
        self.mask_dir = CACHE_ROOT / f"{split_name}_mask"
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _load(self, cache_dir, idx):
        return Image.open(cache_dir / f"{idx}.png").convert("RGB")

    def __getitem__(self, idx):
        full_image = self._load(self.full_dir, idx)
        crop_image = self._load(self.crop_dir, idx)
        mask_image = self._load(self.mask_dir, idx)
        label = int(self.df.loc[idx, "pathology"])

        images = [full_image, crop_image, mask_image]
        images = [TF.resize(im, [RESIZE_SIZE, RESIZE_SIZE]) for im in images]

        if self.augment:
            if random.random() < 0.5:
                images = [TF.hflip(im) for im in images]
            angle = random.uniform(-ROTATION_DEGREES, ROTATION_DEGREES)
            images = [TF.rotate(im, angle) for im in images]

        # Convert each modality to one channel, then stack -> [3, H, W].
        full_gray = TF.to_tensor(TF.rgb_to_grayscale(images[0], num_output_channels=1))
        crop_gray = TF.to_tensor(TF.rgb_to_grayscale(images[1], num_output_channels=1))
        mask_gray = TF.to_tensor(TF.rgb_to_grayscale(images[2], num_output_channels=1))
        x = torch.cat([full_gray, crop_gray, mask_gray], dim=0)

        x = TF.normalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        return x, label


train_dataset = CBISThreeInputCachedDataset(train_df, "train", augment=True)
val_dataset = CBISThreeInputCachedDataset(val_df, "val", augment=False)
test_dataset = CBISThreeInputCachedDataset(test_df, "test", augment=False)

x, label = train_dataset[0]
print("input:", x.shape, "label:", label, "channel order: [full, crop, mask]")

## Multi-model training with true channel stacking

This version uses **one ordinary backbone** per model. For each sample, the three
related images are stacked as channels into a single input tensor:

- C0 = full mammogram
- C1 = cropped ROI
- C2 = ROI mask

Steps:
1. Train on `train_df`, monitor on predefined `val_df`, keep best-val-F1 checkpoint
2. Evaluate each final model once on held-out `test_df`
3. Save both result tables to CSV

Compared with a 3-branch design, this avoids tripling backbone parameters and usually
reduces overfitting risk on smaller datasets.

In [ ]:
import copy
import pandas as pd

import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# ---- experiment config ----
MODEL_NAMES = ["resnet50", "densenet121", "efficientnet_b2", "vgg16"]

EPOCHS = 10
BATCH_SIZE = 16
LR = 1e-4
NUM_CLASSES = 2  # binary: pathology 0/1

# Input channel convention for all models in this notebook:
# C0=full mammogram, C1=cropped ROI, C2=ROI mask
CHANNEL_ORDER = "[full, crop, mask]"

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

In [ ]:
def get_model(name, num_classes=NUM_CLASSES, pretrained=True):
    """Return a standard single-backbone classifier for stacked 3-channel inputs."""
    name = name.lower()

    if name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        m = models.resnet50(weights=weights)
        in_features = m.fc.in_features
        m.fc = nn.Linear(in_features, num_classes)

    elif name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        m = models.densenet121(weights=weights)
        in_features = m.classifier.in_features
        m.classifier = nn.Linear(in_features, num_classes)

    elif name == "efficientnet_b2":
        weights = models.EfficientNet_B2_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b2(weights=weights)
        in_features = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_features, num_classes)

    elif name == "vgg16":
        weights = models.VGG16_Weights.DEFAULT if pretrained else None
        m = models.vgg16(weights=weights)
        in_features = m.classifier[6].in_features
        m.classifier[6] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unknown model name: {name}")

    return m.to(device)

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    try:
        auc = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc = float("nan")  # happens if a fold/batch has only one class present

    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1, "auc": auc}


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * labels.size(0)

        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = avg_loss

    return metrics, all_labels, all_preds, all_probs

### Stage 1 — Train on train_df, monitor on val_df

This notebook now uses your predefined split files directly:
- `train_df` for optimization
- `val_df` for checkpoint selection
- `test_df` for final one-time reporting

In [ ]:
print(f"train images: {len(train_dataset)} | val images: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                           persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                         persistent_workers=(NUM_WORKERS > 0))

In [ ]:
def train_model_single_split(model_name, epochs=EPOCHS, lr=LR):
    model = get_model(model_name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics, _, _, _ = evaluate(model, val_loader, criterion)
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

        print(f"[{model_name}] epoch {epoch}/{epochs} | train_loss {train_loss:.4f} "
              f"| val_loss {val_metrics['loss']:.4f} | val_acc {val_metrics['accuracy']:.4f} "
              f"| val_f1 {val_metrics['f1']:.4f} | val_auc {val_metrics['auc']:.4f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

### Stage 2 — Evaluate once on the held-out test set

Each model is evaluated on `test_df` exactly once, using its best-val-F1 checkpoint
from Stage 1. These are the numbers to report as your final results.

In [ ]:
val_histories = {}
final_results = {}
trained_models = {}

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          persistent_workers=(NUM_WORKERS > 0))
criterion = nn.CrossEntropyLoss()

for model_name in MODEL_NAMES:
    print(f"\n===== {model_name}: train + validate (3-input) =====")
    model, history = train_model_single_split(model_name)
    trained_models[model_name] = model
    val_histories[model_name] = history

    test_metrics, y_true, y_pred, y_proba = evaluate(model, test_loader, criterion)
    print(f"[{model_name}] TEST metrics: {test_metrics}")
    final_results[model_name] = test_metrics

    torch.save(model.state_dict(), f"{model_name}_three_input_final.pt")

test_results_df = pd.DataFrame(final_results).T
test_results_df

### Save results

In [ ]:
test_results_df.to_csv("three_input_test_results.csv")

for model_name, history in val_histories.items():
    history.to_csv(f"{model_name}_three_input_val_history.csv", index=False)

print("Saved three_input_test_results.csv and per-model val history CSVs.")